# 🏥 Welkom bij de Introductie AI Workshop!
**Vandaag ga je zélf een AI model trainen!**

In het ziekenhuis helpt AI steeds vaker bij bijvoorbeeld het scannen van röntgenfoto's. Maar hoe leert een computer eigenlijk zoiets? Vandaag halen we de magie eruit en houden we het simpel: we gaan een AI trainen om het verschil te zien tussen **katten en honden**.

**Hoe werkt dit document?**
Dit is een interactief notitieblok. Je leest de uitleg en drukt daarna op de 'Play' knop: image.png
 bij het code-blok eronder (of gebruik `Shift + Enter`) om de computer een taak te laten uitvoeren. Laten we beginnen!

## 🛠️ Stap 1: De Gereedschapskist Openen
Voordat we kunnen beginnen, moeten we wat hulpmiddelen klaarzetten. Zie dit als het klaarzetten van je instrumenten en protocollen voor een ingreep. We laden hier "pakketjes" in die de computer helpen om plaatjes te openen en slimme berekeningen te maken.

In [ ]:
import os
import tarfile
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import random

from PIL import Image
from sklearn.model_selection import train_test_split

print("Klaar!")

## 📸 Stap 2: De Patiëntendossiers (Foto's) Ophalen
Een arts leert in opleiding door heel veel voorbeelden te zien. Voor AI is dat precies hetzelfde! Om onze AI te trainen, hebben we een grote map met duizenden foto's van honden en katten nodig. We downloaden deze map nu automatisch van het internet. Het downloaden kan een paar minuten duren.

In [ ]:
images_url = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"

if not os.path.exists("images"):
    print("Downloading dataset...")
    urllib.request.urlretrieve(images_url, "images.tar.gz")

    with tarfile.open("images.tar.gz") as tar:
        tar.extractall()

print("Dataset klaar")

## ✂️ Stap 3: De Gegevens Voorbereiden
Computers hebben geen ogen; ze zien alleen wiskundige getallen. Om de AI goed te laten leren, moeten we de foto's netjes ordenen:
1. We kiezen **1000 katten** en **1000 honden**.
2. We knippen alle foto's op exact hetzelfde formaat (128 bij 128 pixels).
3. We zetten de kleuren om in getallen tussen de 0 en 1, zodat de AI er straks makkelijk mee kan rekenen.

In [ ]:
IMAGE_SIZE = 128
MAX_PER_CLASS = 2000

X = []
y = []

# Haal alle jpg bestanden op
all_files = [f for f in os.listdir("images") if f.endswith(".jpg")]

# BELANGRIJK: Shuffle de lijst zodat je een mix van alle rassen krijgt!
random.shuffle(all_files)

cat_count = 0
dog_count = 0

for file in all_files:
    # Katten beginnen met hoofdletter, honden met kleine letter
    label = 0 if file[0].isupper() else 1

    if label == 0 and cat_count >= MAX_PER_CLASS:
        continue
    if label == 1 and dog_count >= MAX_PER_CLASS:
        continue

    path = os.path.join("images", file)

    img = Image.open(path).convert("RGB")
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE))

    X.append(np.array(img))
    y.append(label)

    if label == 0:
        cat_count += 1
    else:
        dog_count += 1

# BELANGRIJK: Normaliseer de pixels naar waarden tussen 0 en 1
X = np.array(X) / 255.0
y = np.array(y)

print("Klaar!")

## 👀 Stap 4: Even Spieken
Zijn onze data en foto's goed ingeladen? Laten we er willekeurig 9 uit de stapel pakken om te kijken hoe de computer ze heeft klaargezet. Ze zien er misschien een beetje wazig uit, omdat we ze expres heel klein hebben gemaakt (128x128)!

In [ ]:
plt.figure(figsize=(8,8))

for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(X[i])
    plt.title("Hond" if y[i] else "Kat")
    plt.axis("off")

plt.tight_layout(pad=2.0)
plt.show()

## 📚 Stap 5: Studiemateriaal vs. Het Examen
Als we de AI álle foto's laten zien tijdens het trainen, leert hij ze misschien gewoon uit zijn hoofd. Dat is niet de bedoeling! Daarom splitsen we onze data op in twee groepen:
* **Trainingsdata (80%):** Het studiemateriaal. Hiermee gaat de AI straks oefenen.
* **Testdata (20%):** Het eindexamen. Deze foto's houden we geheim tot het einde, om te testen of de AI écht het verschil tussen een kat en een hond snapt.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Klaar!")

## 🧠 Stap 6: Het AI-Brein Bouwen
Nu gaan we de AI zelf ontwerpen! We bouwen een 'Neuraal Netwerk'.  

Zie dit als het bouwen van een visueel systeem, van het oog naar de hersenen.
* De eerste lagen ("Conv2D") zijn als de ogen: ze zoeken naar simpele dingen zoals randjes, vacht of kleuren.
* De diepere lagen plakken die informatie samen tot complexe vormen, zoals een snuit of een oor.
* De laatste laag neemt de beslissing: "Is dit een kat of een hond?"

In [41]:
model = tf.keras.Sequential([
    # Blok 1
    tf.keras.layers.Conv2D(16, (3,3), activation="relu", input_shape=(128, 128, 3)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    # Blok 2
    tf.keras.layers.Conv2D(32, (3,3), activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    # Blok 3
    tf.keras.layers.Conv2D(64, (3,3), activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    # Classificatie
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

## 🎯 Stap 7: De Regels van het Spel
We hebben het brein gebouwd, maar de AI weet nog niet *hoe* het moet leren. Hier stellen we het doel in. We vertellen de AI: *"Elke keer dat je een foto verkeerd inschat, krijg je strafpunten (Loss). Probeer dit getal zo laag mogelijk te maken, zodat je nauwkeurigheid (Accuracy) stijgt!"*

In [42]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## 👨‍🎓 Stap 8: Naar School! (De Training)
Dit is het coolste (maar ook het langste) onderdeel: de AI gaat leren!

* **Epoch:** Eén 'epoch' betekent dat de AI de hele stapel studiefoto's één keer heeft doorgekeken.
* We laten hem **10 keer** door de stof heen gaan (10 epochs).

Kijk goed naar de getallen die straks verschijnen. Als het goed is zie je de `accuracy`
(nauwkeurigheid) per ronde steeds een beetje stijgen. De computer wordt letterlijk slimmer
waar je bij zit!

> 💡 **Leuk om te weten:** Wij gebruiken vandaag 10 epochs op een simpel model.
> In de echte wereld, denk aan AI die röntgenfoto's leest of tumoren herkent,
> worden modellen getraind met **honderden tot duizenden epochs**, op computers die
> daar **dagen, weken of zelfs maanden** over doen. Wat wij hier in een paar minuten
> doen is dus eigenlijk een miniatuurversie van datzelfde proces!

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32
)

## 📝 Stap 9: Het Eindexamen
De school is uit! Hoe goed is onze AI nu echt in de echte wereld? We laten hem het eindexamen doen op de 20% foto's die hij nog nooit eerder in zijn leven heeft gezien.
Een accuracy van `1.00` is een 10. Alles boven de 0.75 (75% goed) is voor dit simpele experiment een waanzinnig knappe prestatie!

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)

print(f"Accuracy: {accuracy:.2f}")

## 🔮 Stap 10: De AI in de Praktijk Zien!
Cijfers zijn leuk, maar we willen het natuurlijk met eigen ogen zien werken. Laten we 9 willekeurige foto's uit het eindexamen pakken en de AI vragen wat hij ziet.

* **Pred:** Wat de AI voorspelt *(Prediction)*.
* **True:** Wat de foto in werkelijkheid is.

Kijk maar mee hoeveel hij er goed heeft!

In [ ]:
predictions = model.predict(X_test[:9])

plt.figure(figsize=(8,8))

for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(X_test[i])

    pred_label = "Hond" if predictions[i] > 0.5 else "Kat"
    true_label = "Hond" if y_test[i] == 1 else "Kat"

    plt.title(f"Voorspeld: {pred_label}\nDaadwerkelijk: {true_label}")
    plt.axis("off")

plt.tight_layout(pad=2.0)
plt.show()

## 🤞 Stap 11: Test de AI Zelf!
Nu is het jouw beurt! We pakken een willekeurige foto uit de dataset die de AI
nog niet heeft gezien.

**Run het eerste blok** om een mysteriefoto te zien. Kun jij al zien of het een
kat of een hond is?

**Run daarna het tweede blok** om te kijken of de AI het met je eens is!

In [ ]:
# Pak een willekeurige foto uit de testset
random_index = random.randint(0, len(X_test) - 1)
mystery_image = X_test[random_index]

plt.figure(figsize=(4, 4))
plt.imshow(mystery_image)
plt.title("Wat denk jij? Kat of Hond?", fontsize=14)
plt.axis("off")
plt.show()

In [ ]:
# Laat de AI voorspellen
prediction = model.predict(mystery_image.reshape(1, IMAGE_SIZE, IMAGE_SIZE, 3))
confidence = prediction[0][0]

ai_guess = "Hond" if confidence > 0.5 else "Kat"
ai_confidence = confidence if confidence > 0.5 else 1 - confidence

true_answer = "Hond" if y_test[random_index] == 1 else "Kat"

print(f"De AI denkt:     {ai_guess}")
print(f"Zekerheid:       {ai_confidence:.0%}")
print(f"Het echte antwoord: {true_answer}")
print("=" * 40)

if ai_guess == true_answer:
    print("\nDe AI had gelijk! ✅")
else:
    print("\nOeps! De AI zat ernaast! ❌")

print("\nTip: Run Stap 11 opnieuw voor een nieuwe foto!")